# 03 — Construção dos perfis estruturados para os modelos Qwen

Este notebook executa a etapa de pré-processamento que transforma as publicações filtradas da coleção LExR em perfis estruturados por pesquisador para uso posterior pelos modelos Qwen.

A lógica reutilizável está em `src/preprocessing/build_profiles_qwen.py`.

O procedimento preserva a estrutura individual das publicações e mantém separadamente:

- título;
- palavras-chave;
- resumo.

A limpeza é leve, sem conversão para minúsculas e sem remoção de acentos.

> Este notebook apenas constrói os perfis de entrada. A montagem do prompt, o limite de publicações, o truncamento por tokens e a inferência dos modelos são realizados em etapas posteriores.

## 1. Configuração do projeto

Os caminhos onde estão os arquivos processados da coleção.

In [ ]:
from pathlib import Path
import json
import sys

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next((p for p in candidates if (p / "src").exists()), current)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing.build_profiles_qwen import gerar_perfis_qwen

print(f"Raiz do projeto: {PROJECT_ROOT}")

## 2. Caminhos

In [ ]:
DATA_DIR = PROJECT_ROOT / "data" / "processed"

DOCUMENTS = DATA_DIR / "filtered_documents.json"
QRELS = DATA_DIR / "ground_truth" / "LExR-prof-qrels_filtrado"
OUTPUT = DATA_DIR / "perfis_estruturados_qwen.json"

for name, path in {
    "filtered_documents.json": DOCUMENTS,
    "LExR-prof-qrels_filtrado": QRELS,
}.items():
    print(f"{name:32s} -> {'OK' if path.exists() else 'não encontrado'}")

## 3. Geração dos perfis

Cada publicação é associada a todos os coautores presentes no conjunto de qrels filtrados.

São mantidas publicações com pelo menos título ou resumo preenchido. As palavras-chave são reunidas em uma única string separada por vírgulas, reproduzindo o formato utilizado nos notebooks de inferência.

In [ ]:
perfis = gerar_perfis_qwen(
    documentos=DOCUMENTS,
    qrels=QRELS,
    saida=OUTPUT,
)

## 4. Verificação da saída

In [ ]:
print(f"Autores com perfil: {len(perfis):,}")
print(f"Entradas autor-publicação: {sum(len(v) for v in perfis.values()):,}")

for author_id, publications in list(perfis.items())[:3]:
    print("\n" + "=" * 80)
    print(f"Autor: {author_id}")
    print(f"Número de publicações: {len(publications)}")
    if publications:
        print(json.dumps(publications[0], ensure_ascii=False, indent=2))

## 5. Arquivo produzido

O notebook gera:

- `perfis_estruturados_qwen.json`

Esse arquivo é utilizado posteriormente na construção dos prompts e na inferência dos modelos Qwen.

A etapa de construção não aplica ordenação adicional por ano nem limita o perfil a 50 publicações. Essas condições devem ser tratadas na etapa correspondente à montagem da entrada do modelo, conforme o experimento executado.